# Attack Path Intelligence Engine

**Graph-Based Risk Analysis and Remediation**

This notebook independently demonstrates the core analysis using a synthetic educational dataset. It validates input, builds a directed graph, discovers bounded paths, calculates transparent triage estimates, simulates remediation on a copy, compares results, draws visuals, and runs the project tests. Scores are not probabilities of compromise. No systems are accessed or changed.

## Dependencies

Install from the project folder with `python -m pip install -r requirements.txt`. Python 3.11 or newer is recommended. The notebook needs the packages declared in that file.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import networkx as nx
import plotly.express as px
ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src').exists())
sys.path.insert(0, str(ROOT))
print('Project:', ROOT)
print('Python:', sys.version.split()[0])

## Load and validate the synthetic dataset

The schema describes directed relationships. Passwords and tokens are not collected. The loader enforces size, row, schema, and value constraints.

In [ ]:
from src.data_loader import load_relationships
from src.graph_builder import build_graph, critical_assets, low_privilege_starts
from src.path_analyzer import discover_paths, explain_path
from src.risk_engine import score_path
from src.remediation import simulate_remediation, recommendation
from src.report_generator import make_report
data_path = ROOT / 'data' / 'sample_environment.csv'
relationships = load_relationships(data_path)
print(f'{len(relationships)} validated unique relationship rows')
display(relationships.head(10))

## Build the directed graph

CSV rows become directed edges with relationship and weakness attributes. Candidate starts are low- or medium-privilege identities; targets are asset nodes.

In [ ]:
graph = build_graph(relationships)
print(f'{graph.number_of_nodes()} nodes, {graph.number_of_edges()} edges')
print('Candidate starting identities:', low_privilege_starts(graph))
print('Asset targets:', critical_assets(graph))

## Discover bounded attack paths

The search enumerates simple directed paths up to six edges and returns no more than 100 results. A path describes modeled reachability only.

In [ ]:
paths = discover_paths(graph, max_length=6, max_paths=100)
print('Discovered paths:', len(paths))
for path in paths:
    print(explain_path(path))
    print('  route:', ' -> '.join(path['nodes']))
    print('  relationships:', path['relationships'])

## Risk scoring

The score weights asset criticality (25), starting privilege (15), maximum weakness severity (30), business impact (20), and path proximity (10). Categories are Low (0–34), Medium (35–59), High (60–79), and Critical (80–100). This is a prioritization estimate, not a validated probability.

In [ ]:
path_table = pd.DataFrame([{'start': p['start'], 'target': p['target'], 'route': ' -> '.join(p['nodes']), 'length': p['length'], 'score': p['risk']['score'], 'category': p['risk']['category'], 'factors': p['risk']['factors'], 'weaknesses': ', '.join(p['weaknesses'])} for p in paths])
display(path_table)

## Simulated remediation and comparison

The selected weakness is removed from a separate DataFrame copy. The original frame remains unchanged and no real configuration change is made.

In [ ]:
weakness_to_fix = 'overbroad_share_permission'
comparison = simulate_remediation(relationships, weakness_to_fix, max_length=6, max_paths=100)
print('Original rows:', len(comparison['before_frame']), 'Modified analysis rows:', len(comparison['after_frame']))
print('Paths before:', len(comparison['before_paths']))
print('Paths after:', len(comparison['after_paths']))
print('Paths removed:', len(comparison['removed_paths']))
print('Remaining high/critical paths:', sum(p['risk']['category'] in {'High','Critical'} for p in comparison['after_paths']))
assert relationships.equals(comparison['before_frame'])
for p in comparison['removed_paths']:
    print('Removed route:', ' -> '.join(p['nodes']))

## Visualizations

The chart summarizes illustrative path risk categories from the bundled sample.

In [ ]:
if len(path_table):
    fig = px.bar(path_table['category'].value_counts().reindex(['Critical','High','Medium','Low'], fill_value=0).rename_axis('Risk').reset_index(name='Paths'), x='Risk', y='Paths', title='Sample path risk distribution', color='Risk', color_discrete_map={'Critical':'#ff5d5d','High':'#ff9f43','Medium':'#f5ca52','Low':'#55c595'})
    fig.show()
else:
    print('No paths found for this dataset.')

## Test execution

The project tests cover valid, invalid, empty, and malformed data, graph building, path bounds, risk scoring, and the remediation effect. Results below are produced by this environment when the notebook is executed.

In [ ]:
import subprocess
result = subprocess.run([sys.executable, '-m', 'pytest', '-q'], cwd=ROOT, text=True, capture_output=True)
print(result.stdout)
if result.stderr: print(result.stderr)
assert result.returncode == 0, 'Pytest failed; inspect output above.'

## Conclusions and limitations

This implementation makes supplied relationships and multi-step routes inspectable, provides a bounded search, and compares a hypothetical weakness removal. Its reachability depends on complete and accurate inputs. The scoring weights are heuristic and have not been validated against incidents or business loss. The tool does not discover systems, authenticate, exploit, or verify remediation. Use synthetic or authorized data only.

## Report object preview

The dashboard additionally offers JSON and CSV downloads with UTC timestamp, dataset name, path details, score factors, recommendations, and comparison summary.

In [ ]:
report = make_report(relationships, paths, data_path.name, comparison)
print('Generated UTC:', report['generated_at_utc'])
print('Summary:', report['environment_summary'])
print('Comparison:', report['comparison'])